In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Tuple

@dataclass
class ModelArgs:
    dim: int = 512
    n_heads: int = 8
    max_seq_len: int = 1024
    vocab_size: int = 1000

# 1. 生成旋转矩阵
def precompute_freqs_cis(dim: int, seq_len: int, theta: float = 10000.0):
    # dim 应该是 head_dim，而不是总的 embedding dim
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(seq_len, device=freqs.device)
    freqs = torch.outer(t, freqs).float()  # [seq_len, head_dim // 2]
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs) 
    return freqs_cis

# 2. 旋转位置编码计算
def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    # xq.shape = [batch_size, seq_len, n_heads, head_dim]
    
    # 重塑为复数形式: [batch, seq, heads, head_dim/2, 2] -> view_as_complex -> [batch, seq, heads, head_dim/2]
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    
    # 调整 freqs_cis 形状以支持广播
    # freqs_cis 原始: [seq_len, head_dim // 2]
    # 目标广播: [1, seq_len, 1, head_dim // 2]
    freqs_cis = freqs_cis.view(1, xq_.size(1), 1, xq_.size(-1))
    
    # 应用旋转
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    
    return xq_out.type_as(xq), xk_out.type_as(xk)

class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads
        
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)
        
        # 预计算旋转矩阵并注册为 buffer (不会被优化器更新，但会随模型保存)
        # 注意传入的是 self.head_dim
        freqs_cis = precompute_freqs_cis(self.head_dim, args.max_seq_len * 2)
        self.register_buffer("freqs_cis", freqs_cis)

    def forward(self, x: torch.Tensor):
        bsz, seqlen, _ = x.shape
        
        # 1. 投影
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # 2. 拆分多头 [batch, seq, n_heads, head_dim]
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_heads, self.head_dim)

        # 3. 应用 RoPE
        # 切片截取当前序列长度对应的频率矩阵
        freqs_cis = self.freqs_cis[:seqlen] 
        xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)

        # 4. 转置以进行 Attention 计算 [batch, n_heads, seq, head_dim]
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)
        
        # 5. 计算 Attention Scores
        # scores.shape = (bsz, n_heads, seqlen, seqlen)
        scores = torch.matmul(xq, xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        
        # 6. 计算 Output
        output = torch.matmul(scores, xv)  # (bsz, n_heads, seqlen, head_dim)
        
        # 7. 还原形状并投影输出
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.wo(output)

# --- 测试代码 ---
if __name__ == "__main__":
    args = ModelArgs()
    model = Attention(args)
    
    # 创建模拟输入 [batch_size, seq_len, dim]
    x = torch.randn(2, 32, args.dim)
    
    try:
        y = model(x)
        print(f"输入形状: {x.shape}")
        print(f"输出形状: {y.shape}")
        print("代码运行成功！")
    except Exception as e:
        print(f"运行出错: {e}")

    print(model.freqs_cis)

输入形状: torch.Size([2, 32, 512])
输出形状: torch.Size([2, 32, 512])
代码运行成功！
tensor([[ 1.0000+0.0000e+00j,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          ...,  1.0000+0.0000e+00j,  1.0000+0.0000e+00j,
          1.0000+0.0000e+00j],
        [ 0.5403+8.4147e-01j,  0.7318+6.8156e-01j,  0.8460+5.3317e-01j,
          ...,  1.0000+2.3714e-04j,  1.0000+1.7783e-04j,
          1.0000+1.3335e-04j],
        [-0.4161+9.0930e-01j,  0.0709+9.9748e-01j,  0.4315+9.0213e-01j,
          ...,  1.0000+4.7427e-04j,  1.0000+3.5566e-04j,
          1.0000+2.6670e-04j],
        ...,
        [-0.9844+1.7590e-01j,  0.9062+4.2275e-01j,  0.9864+1.6438e-01j,
          ...,  0.8847+4.6616e-01j,  0.9346+3.5570e-01j,
          0.9630+2.6934e-01j],
        [-0.6799-7.3331e-01j,  0.3750+9.2701e-01j,  0.7468+6.6501e-01j,
          ...,  0.8846+4.6637e-01j,  0.9345+3.5586e-01j,
          0.9630+2.6947e-01j],
        [ 0.2497-9.6832e-01j, -0.3574+9.3397e-01j,  0.2774+9.6077e-01j,
          ...,  0.8845+4.6658e-01j,  0.9345+3

In [8]:
import torch

# 设置随机种子保证可复现
torch.manual_seed(42)

# ==========================================
# 准备工作：生成频率 (Common)
# ==========================================
def precompute_freqs_cis(head_dim: int, seq_len: int, theta: float = 10000.0):
    # 计算角度 theta_i
    # 注意：根据 LLaMA 实现，频率只生成 dim/2 个
    # head_dim = 64 -> freqs last dim = 32
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    
    # outer product: [seq_len, head_dim/2]
    freqs = torch.outer(t, freqs)
    return freqs

# ==========================================
# 版本 A: LLaMA/Qwen 官方实数实现 (Real)
# ==========================================
def rotate_half(x):
    """(x1, x2) -> (-x2, x1)"""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope_real_llama(x, freqs):
    # x: [bs, seq, heads, dim]
    # freqs: [seq, dim/2]
    
    # 1. 为了让 cos/sin 能跟 x (dim) 对应，需要把 freqs (dim/2) 复制拼接一次
    # [seq, dim/2] -> [seq, dim]
    freqs_full = torch.cat((freqs, freqs), dim=-1)
    
    # 2. 调整形状以广播: [1, seq, 1, dim]
    freqs_full = freqs_full.view(1, x.shape[1], 1, x.shape[-1])
    
    cos = freqs_full.cos()
    sin = freqs_full.sin()
    
    # 3. 核心公式: x * cos + rotate_half(x) * sin
    return (x * cos) + (rotate_half(x) * sin)

# ==========================================
# 版本 B: 复数等价实现 (Complex - Fixed)
# ==========================================
def apply_rope_complex_llama(x, freqs):
    # x: [bs, seq, heads, dim] (e.g. dim=64)
    dim = x.shape[-1]
    
    # 1. 构造复数
    # LLaMA 的逻辑是: index i 和 index i + dim/2 是一对
    # x_real: [..., 32], x_imag: [..., 32]
    x_real = x[..., : dim // 2]
    x_imag = x[..., dim // 2 :]
    x_complex = torch.complex(x_real, x_imag) # shape: [bs, seq, heads, 32]
    
    # 2. 构造复数旋转因子
    # freqs: [seq, 32]
    # 【修复点】这里 reshape 的最后一维必须是 32 (即 dim // 2)，而不是 64
    freqs = freqs.view(1, x.shape[1], 1, dim // 2) 
    
    # 生成复数旋转子 (模长为1, 角度为 freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    
    # 3. 复数乘法 (旋转)
    # [bs, seq, heads, 32] * [1, seq, 1, 32]
    x_out_complex = x_complex * freqs_cis
    
    # 4. 还原回实数 (拼接实部和虚部)
    # 对应 LLaMA 的布局：先放实部(前半截)，再放虚部(后半截)
    return torch.cat([x_out_complex.real, x_out_complex.imag], dim=-1)

# ==========================================
# 比较测试
# ==========================================
def run_comparison():
    # 参数设置
    bs, seq_len, n_heads, head_dim = 2, 128, 4, 64
    
    # 随机输入
    x = torch.randn(bs, seq_len, n_heads, head_dim)
    
    # 预计算频率 (输出 shape: [128, 32])
    freqs = precompute_freqs_cis(head_dim, seq_len)
    
    # 运行两个版本
    output_real = apply_rope_real_llama(x.clone(), freqs)
    output_complex = apply_rope_complex_llama(x.clone(), freqs)
    
    # 检查误差
    is_close = torch.allclose(output_real, output_complex, atol=1e-6)
    max_diff = (output_real - output_complex).abs().max().item()
    
    print(f"输入形状: {x.shape}")
    print(f"Freqs形状: {freqs.shape}")
    print(f"版本 A (实数/Llama) 输出形状: {output_real.shape}")
    print(f"版本 B (复数/Complex) 输出形状: {output_complex.shape}")
    print("-" * 30)
    print(f"最大差异 (Max Difference): {max_diff:.8f}")
    print(f"结果是否一致 (torch.allclose): {is_close}")
    
    if is_close:
        print("\n✅ 验证成功：复数写法与 LLaMA 实数写法完全等价！")
    else:
        print("\n❌ 验证失败：结果不一致。")

if __name__ == "__main__":
    run_comparison()

输入形状: torch.Size([2, 128, 4, 64])
Freqs形状: torch.Size([128, 32])
版本 A (实数/Llama) 输出形状: torch.Size([2, 128, 4, 64])
版本 B (复数/Complex) 输出形状: torch.Size([2, 128, 4, 64])
------------------------------
最大差异 (Max Difference): 0.00000048
结果是否一致 (torch.allclose): True

✅ 验证成功：复数写法与 LLaMA 实数写法完全等价！


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Tuple, Optional

@dataclass
class ModelArgs:
    dim: int = 512
    n_heads: int = 8
    max_seq_len: int = 1024
    vocab_size: int = 1000

# 1. 生成 Cos 和 Sin 表 (LLaMA/Qwen 风格)
def precompute_cos_sin(dim: int, seq_len: int, theta: float = 10000.0):
    # 计算 theta_i，维度是 head_dim // 2
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(seq_len, device=inv_freq.device)
    
    # 计算 outer product: [seq_len, head_dim // 2]
    freqs = torch.outer(t, inv_freq)
    
    # 关键差异：LLaMA 逻辑是将 freqs 拼接两次，以适配 rotate_half 的对半切分
    # 结果 shape: [seq_len, head_dim]
    emb = torch.cat((freqs, freqs), dim=-1)
    
    # 返回 cos 和 sin (实数)
    return emb.cos(), emb.sin()

# 2. 定义 rotate_half 操作
def rotate_half(x: torch.Tensor):
    """
    将输入张量的最后一维分成两半，并进行交换和符号变换。
    x: [..., d] -> x1: [..., d/2], x2: [..., d/2]
    out: cat(-x2, x1)
    """
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# 3. 旋转位置编码计算 (实数域)
def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    # xq.shape = [batch_size, seq_len, n_heads, head_dim]
    # cos, sin 原始 shape = [seq_len, head_dim]
    
    # 调整 cos, sin 形状以支持广播: [seq_len, head_dim] -> [1, seq_len, 1, head_dim]
    # 注意：这里假设 seq_len 维度已经切片匹配了
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    
    # 应用公式: (x * cos) + (rotate_half(x) * sin)
    xq_embed = (xq * cos) + (rotate_half(xq) * sin)
    xk_embed = (xk * cos) + (rotate_half(xk) * sin)
    
    return xq_embed.type_as(xq), xk_embed.type_as(xk)

class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads
        
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)
        
        # 预计算 Cos 和 Sin 并注册为 buffer
        # 这里传入 self.head_dim
        cos, sin = precompute_cos_sin(self.head_dim, args.max_seq_len * 2)
        self.register_buffer("cos_cached", cos)
        self.register_buffer("sin_cached", sin)

    def forward(self, x: torch.Tensor):
        bsz, seqlen, _ = x.shape
        
        # 1. 投影
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # 2. 拆分多头 [batch, seq, n_heads, head_dim]
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_heads, self.head_dim)

        # 3. 应用 RoPE (使用 LLaMA/Qwen 风格)
        # 根据当前 seqlen 切片
        cos = self.cos_cached[:seqlen] 
        sin = self.sin_cached[:seqlen]
        
        xq, xk = apply_rotary_emb(xq, xk, cos=cos, sin=sin)

        # 4. 转置以进行 Attention 计算 [batch, n_heads, seq, head_dim]
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)
        
        # 5. 计算 Attention Scores
        # scores.shape = (bsz, n_heads, seqlen, seqlen)
        scores = torch.matmul(xq, xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        
        # 6. 计算 Output
        output = torch.matmul(scores, xv)  # (bsz, n_heads, seqlen, head_dim)
        
        # 7. 还原形状并投影输出
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.wo(output)

# --- 测试代码 ---
if __name__ == "__main__":
    args = ModelArgs()
    model = Attention(args)
    
    # 创建模拟输入 [batch_size, seq_len, dim]
    x = torch.randn(2, 32, args.dim)
    
    try:
        y = model(x)
        print(f"输入形状: {x.shape}")
        print(f"输出形状: {y.shape}")
        print("代码运行成功！")
        
        # 打印缓存形状验证
        print(f"Cos Cache Shape: {model.cos_cached.shape}")
        
    except Exception as e:
        print(f"运行出错: {e}")
        import traceback
        traceback.print_exc()

输入形状: torch.Size([2, 32, 512])
输出形状: torch.Size([2, 32, 512])
代码运行成功！
Cos Cache Shape: torch.Size([2048, 64])
